L2-regularized logistic regression for binary or multiclass classification; trains a model (on `train.txt`), optimizes L2 regularization strength on `dev.txt`, and evaluates performance on `test.txt`.  Reports test accuracy with 95% confidence intervals and prints out the strongest coefficients for each class.

## Import

In [3]:
!pip install imbalanced-learn nrclex

Defaulting to user installation because normal site-packages is not writeable


In [1]:
from scipy import sparse
from sklearn import linear_model
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from imblearn.over_sampling import SMOTE
from collections import Counter
from itertools import product
import numpy as np
import operator
import nltk
import json
import math
import os
from scipy.stats import norm
import string
from tqdm import tqdm

# Load NRC lexicon directly to avoid nrclex path resolution bug
_nrclex_path = os.path.join(os.path.dirname(__import__('nrclex').__file__), 'data', 'nrc_en.json')
with open(_nrclex_path) as f:
    NRC_LEXICON = json.load(f)

In [2]:
!python -m nltk.downloader punkt

<frozen runpy>:128: RuntimeWarning: 'nltk.downloader' found in sys.modules after import of package 'nltk', but prior to execution of 'nltk.downloader'; this may result in unpredictable behaviour
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\great\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## File Loading

In [2]:
def load_data(filename):
    X = []
    Y = []
    i = 0
    with open(filename, encoding="utf-8") as file:
        for line in file:
            if i == 0:
                i += 1
                continue
            cols = line.split("\t")
            idd = cols[0]
            label = cols[1].lstrip().rstrip()
            text = cols[2].lstrip().rstrip()

            X.append(text)
            Y.append(label)

    return X, Y


## Classifier Class

In [3]:
class Classifier:
    def __init__(self, feature_methods: list, trainX, trainY, devX, devY, testX, testY):
        self.feature_vocab = {}
        self.feature_methods = feature_methods
        self.min_feature_count=2
        self.log_reg = None

        self.trainY=trainY
        self.devY=devY
        self.testY=testY
        
        self.trainX = self.process(trainX, training=True)
        self.devX = self.process(devX, training=False)
        self.testX = self.process(testX, training=False)

    # Featurize entire dataset
    def featurize(self, data):
        featurized_data = []
        for text in data:
            combined_features = {}
            for feature_method in self.feature_methods:
                feats = feature_method(text)
                combined_features.update(feats)  # Merge dictionaries
            featurized_data.append(combined_features)
        return featurized_data

    # Read dataset and returned featurized representation as sparse matrix + label array
    def process(self, X_data, training = False):
        data = self.featurize(X_data)

        if training:
            fid = 0
            feature_doc_count = Counter()
            for feats in data:
                for feat in feats.keys():  # Use keys of the dictionary
                    feature_doc_count[feat] += 1

            for feat in feature_doc_count:
                if feature_doc_count[feat] >= self.min_feature_count:
                    self.feature_vocab[feat] = fid
                    fid += 1

        F = len(self.feature_vocab)
        D = len(data)
        X = sparse.dok_matrix((D, F))
        for idx, feats in enumerate(data):
            for feat, value in feats.items():  # Use key-value pairs
                if feat in self.feature_vocab:
                    X[idx, self.feature_vocab[feat]] = value

        return X

    # Train model and evaluate on held-out data
    def train(self):
        min_class_count = min(Counter(self.trainY).values())
        k_neighbors = min(5, min_class_count - 1)

        if k_neighbors >= 1:
            smote = SMOTE(k_neighbors=k_neighbors)
            trainX_resampled, trainY_resampled = smote.fit_resample(self.trainX, self.trainY)
        else:
            trainX_resampled, trainY_resampled = self.trainX, self.trainY

        C_vals = [0.01, 0.1, 1, 10, 100]
        # Valid combinations: l1 only works with squared_hinge
        param_grid = [
            (C, penalty, loss)
            for C, penalty, loss in product(C_vals, ['l1', 'l2'], ['hinge', 'squared_hinge'])
            if not (penalty == 'l1' and loss == 'hinge')
        ]

        best_dev_accuracy=0
        best_model=None
        for C, penalty, loss in param_grid:
            self.log_reg = OneVsRestClassifier(
                LinearSVC(C=C, penalty=penalty, loss=loss, max_iter=1000000, class_weight='balanced')
            )
            self.log_reg.fit(trainX_resampled, trainY_resampled)
            development_accuracy = self.log_reg.score(self.devX, self.devY)
            if development_accuracy > best_dev_accuracy:
                best_dev_accuracy=development_accuracy
                best_model=self.log_reg

        self.log_reg=best_model
        

    def test(self):
        return self.log_reg.score(self.testX, self.testY)

    def printWeights(self, n=10):
        reverse_vocab=[None]*len(self.log_reg.estimators_[0].coef_[0])
        for k in self.feature_vocab:
            reverse_vocab[self.feature_vocab[k]]=k

        for i, cat in enumerate(self.log_reg.classes_):
            weights = self.log_reg.estimators_[i].coef_[0]
            for feature, weight in list(reversed(sorted(zip(reverse_vocab, weights), key=operator.itemgetter(1))))[:n]:
                print("%s\t%.3f\t%s" % (cat, weight, feature))
            print()

## Feature Engineering

In [4]:
X, Y = load_data("../adjuticated.txt")
stopwords = set(nltk.corpus.stopwords.words('english'))
punctuation = set(string.punctuation)
punctuation.add("``")
punctuation.add("''")
punctuation.add('’')
punctuation.add('“')
punctuation.add('”')
punctuation.add('—')
punctuation.add("'s")
words_fear = {}
words_name_calling = {}
for text, label in zip(X, Y):
    if label == "Appeal to Fear":
        for word in nltk.word_tokenize(text):
            word=word.lower()
            if word not in stopwords and word not in punctuation:
                if word not in words_fear.keys():
                    words_fear[word]=0
                words_fear[word]+=1
    elif label == "Name Calling":
        for word in nltk.word_tokenize(text):
            word=word.lower()
            if word not in stopwords and word not in punctuation:
                if word not in words_name_calling.keys():
                    words_name_calling[word]=0
                words_name_calling[word]+=1

words_fear = sorted(words_fear.items(), key=operator.itemgetter(1), reverse=True)
words_name_calling = sorted(words_name_calling.items(), key=operator.itemgetter(1), reverse=True)

print("Appeal to Fear:", words_fear[:15], "\nName Calling:", words_name_calling[:15])
print("Classes:", set(Y))


Appeal to Fear: [('act', 23), ('security', 22), ('must', 20), ('congress', 16), ('also', 15), ('nation', 12), ('national', 11), ('health', 11), ('social', 11), ('would', 10), ('ensure', 10), ('tax', 10), ('access', 9), ('care', 9), ('legislation', 9)] 
Name Calling: [('government', 11), ('act', 10), ('federal', 10), ('energy', 9), ('congress', 8), ('president', 8), ('trump', 8), ('funding', 8), ('tax', 7), ('would', 6), ('work', 6), ('time', 6), ('less', 5), ('americans', 5), ('benefits', 5)]
Classes: {'Loaded Language', 'Name Calling', 'No Propaganda', 'Appeal to Fear'}


In [5]:
def binary_bow_featurize(text):
    feats = {}
    words = nltk.word_tokenize(text)

    for word in words:
        word=word.lower()
        feats[word]=1
            
    return feats

def bigram(text):
    feats = {}
    tokens = [w.lower() for w in nltk.word_tokenize(text) if w.isalpha()]
    for i in range(len(tokens) - 1):
        feats[f"{tokens[i]}_{tokens[i+1]}"] = 1

    return feats

def trigram(text):
    feats = {}
    tokens = [w.lower() for w in nltk.word_tokenize(text) if w.isalpha()]
    for i in range(len(tokens) - 2):
        feats[f"{tokens[i]}_{tokens[i+1]}_{tokens[i+2]}"] = 1

    return feats

def quadrigram(text):
    feats = {}
    tokens = [w.lower() for w in nltk.word_tokenize(text) if w.isalpha()]
    for i in range(len(tokens) - 3):
        feats[f"{tokens[i]}_{tokens[i+1]}_{tokens[i+2]}_{tokens[i+3]}"] = 1

    return feats

def appeal_to_fear_featurize(text):
    feats = {}
    words = nltk.word_tokenize(text)

    for word in words:
        word=word.lower()
        if word in ["act", "security", "must", "congress", "also", "nation", "national", "health", "social", "tax", "access", "care", "legislation"]:
            feats[word] = feats.get(word, 0) + 1
            
    return feats

def name_calling_featurize(text):
    feats = {}
    words = nltk.word_tokenize(text)

    for word in words:
        word=word.lower()
        if word in ["government", "act", "federal", "energy", "congress", "president", "trump", "funding", "tax", "americans", "benefits"]:
            feats[word] = feats.get(word, 0) + 1
            
    return feats

def nrc_featurize(text):
    counts = {}
    for word in nltk.word_tokenize(text.lower()):
        for category in NRC_LEXICON.get(word, []):
            counts[category] = counts.get(category, 0) + 1
    total = sum(counts.values()) or 1
    return {f'nrc_{cat}': count / total for cat, count in counts.items()}

def sentence_structure_featurize(text):
    feats = {}
    sentences = nltk.sent_tokenize(text)
    feats['num_sentences'] = len(sentences)
    words = nltk.word_tokenize(text)
    feats['num_words'] = len(words)
    feats['avg_sentence_length'] = len(words) / (len(sentences) or 1)
    return feats

def pronoun_featurize(text):
    words = [w.lower() for w in nltk.word_tokenize(text)]
    total = (len(words) or 1)
    feats = {}
    feats['first_person_pronoun_ratio'] = sum(w in ['i', 'me', 'my', 'mine', 'we', 'us', 'our', 'ours'] for w in words) / total
    feats['second_person_pronoun_ratio'] = sum(w in ['you', 'your', 'yours'] for w in words) / total
    feats['third_person_pronoun_ratio'] = sum(w in ['he', 'him', 'his', 'she', 'her', 'hers', 'they', 'them', 'their', 'theirs'] for w in words) / total
    return feats

In [6]:
def confidence_intervals(accuracy, n, significance_level):
    critical_value=(1-significance_level)/2
    z_alpha=-1*norm.ppf(critical_value)
    se=math.sqrt((accuracy*(1-accuracy))/n)
    return accuracy-(se*z_alpha), accuracy+(se*z_alpha)

## Run Function

In [7]:
def run(trainingFile, devFile, testFile):
    trainX, trainY=load_data(trainingFile)
    devX, devY=load_data(devFile)
    testX, testY=load_data(testFile)

    for feature in tqdm([binary_bow_featurize, bigram, trigram, appeal_to_fear_featurize, name_calling_featurize, nrc_featurize, sentence_structure_featurize, pronoun_featurize]):
        classifier = Classifier([feature], trainX, trainY, devX, devY, testX, testY)
        classifier.train()
        accuracy = classifier.test()
        lower, upper=confidence_intervals(accuracy, len(testY), .95)
        print(f"Test accuracy for '{feature.__name__}': {accuracy:.3f}, 95% CIs: [{lower:.3f} {upper:.3f}]\n")
        
    # simple_classifier.printWeights()

In [8]:
trainingFile = "splits/train.txt"
devFile = "splits/dev.txt"
testFile = "splits/test.txt"
    
run(trainingFile, devFile, testFile)


  0%|          | 0/8 [00:00<?, ?it/s]

 12%|█▎        | 1/8 [00:12<01:30, 12.97s/it]

Test accuracy for 'binary_bow_featurize': 0.530, 95% CIs: [0.432 0.628]



 25%|██▌       | 2/8 [00:17<00:49,  8.17s/it]

Test accuracy for 'bigram': 0.580, 95% CIs: [0.483 0.677]



...

Test accuracy for 'nrc_featurize': 0.420, 95% CIs: [0.323 0.517]



C:\Users\great\AppData\Roaming\Python\Python312\site-packages\sklearn\svm\_base.py:1243: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\great\AppData\Roaming\Python\Python312\site-packages\sklearn\svm\_base.py:1243: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\great\AppData\Roaming\Python\Python312\site-packages\sklearn\svm\_base.py:1243: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\great\AppData\Roaming\Python\Python312\site-packages\sklearn\svm\_base.py:1243: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\great\AppData\Roaming\Python\Python312\site-packages\sklearn\svm\_base.py:1243: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\great\AppData\Roaming\Python\Python3

Test accuracy for 'sentence_structure_featurize': 0.380, 95% CIs: [0.285 0.475]



100%|██████████| 8/8 [02:14<00:00, 16.79s/it]

Test accuracy for 'pronoun_featurize': 0.470, 95% CIs: [0.372 0.568]

